# Datamine PROPER Process Example

This notebook demonstrates how to configure and run the **`proper`** process wrapper in `dmstudio`.

## Process Description

## PROPER

Copy and condition a set of perimeters.

This process is used to:

* Add an **AREA** field.
* Make all perimeters clockwise or anticlockwise.
* Close or open all perimeters.
* Set a minimum or maximum string segment length.
* Reduce the number of points by a percentage.
* Extend or shrink perimeters.

Optionally, the input may be treated as strings as opposed to perimeters. If so parameters **AREA, CLOCKWSE, VPLANE, EXTEND** and **CROSS** are ignored.

The reduce and tolerance options will not remove points that have a valid **TAG**. That is if a field TAG exists and is not set to 0 or missing.

Input perimeters must be planar. If the perimeters are within 6 digit tolerance of either the **XY, XZ** or **YZ** then they will be assumed to lie on the orthogonal plane. The output perimeters will be constant in the third dimension. This is useful if the perimeters have numerical rounding errors produced by prior processing. Non-planar perimeters will not be processed or output.

All perimeters will automatically be checked for crossovers and consecutive duplicate points. The duplicate points will be removed, however, the malformed perimeters will not be processed or output.

After point reduction or extension some perimeters may become malformed. These will be reported and output.

Extra points added by use of the @**EXTEND** or @**DMAX** parameters will have all attribute values set the same as the first point of the original perimeter.

For perimeters that are not on an orthogonal plane clockwise direction will be taken from an orthogonal view plane. All perimeters in parallel planes will have the same clockwise sense.

This process may be run without an output file for checking perimeters and reporting areas. It may not be run in place.

### Input Files:

* **perimin** (String):
  The input perimeter file. The fields required are **XP,YP,ZP,PTN** , and **PVALUE**
  (standard perimeter format). All perimeters in the file will be used. All other fields
  will be copied. Perimeters must be planar.
  Required=Yes

### Output Files:

* **perimout** (String):
  Output perimeter file. Contains all fields from the input file plus optionally

## **AREA**.

  Required=No

### Fields:

### Parameters:

* **mode**:
  For **MODE** =1 only parameters **CLOSE, DMAX, TOL, REDUCE** are used to treat strings.
  =0 : Treat as perimeters. =1 : Treat as strings.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **area**:
  field containing perimeter area creation flag =0 : dont create field **AREA** in
  **PERIMOUT** . =1 : create field **AREA** in **PERIMOUT** .
  Range=0,1
  Values=0,1
  Default=0,1
  Required=No

* **close**:
  =0 : will remove last point of a perimeter if perimeter is closed. . =1 : will add
  first point to end of perimeter if perimeter not closed..
  Range=Undefined
  Values=-,0,1
  Default=0
  Required=No

* **clockwse**:
  =0 : make all perimeters anti-clockwise. =1 : make all perimeters clockwise.
  Range=Undefined
  Values=-,0,1
  Default=Undefined
  Required=No

* **vplane**:
  Viewing plane for clockwise sense for non-orthogonal planes (1). =1 : XY plane from +Z.
  =2 : XZ plane from -Y. =3 : YZ plane from +X.
  Range=1,3
  Values=1,2,3
  Default=1
  Required=No

* **dmax**:
  The maximum chord length used when inserting additional points into long chords.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **tol**:
  Minimum allowable chord length used when removing points. Default is (0) for removal of
  consecutive duplicates.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **reduce**:
  Percentage point reduction 0 to 90 (0).
  Range=0,90
  Values=Undefined
  Default=0
  Required=No

* **extend**:
  +/- perpendicular extension distance (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **cross**:

* **Options** ((0): do not attempt to resolve crossovers in extended perimeters.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **zeroxyz**:
  Internally set the X, Y or Z value of input perimeters to zero. An example of when this
  is useful is if you want to calculate plan areas of 3 dimensional perimeters =(0) : Do
  not internally adjust any coordinates in the input perimeters.. =1 : Internally treat
  all X coordinates in the input strings as zero. =2 : Internally treat all Y coordinates
  in the input strings as zero. =3 : Internally treat all Z coordinates in the input
  strings as zero.
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('proper')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute proper
print("Running proper...")
dm_cmd.proper(
    perimin_i='t_input_file',  # required
    zeroxyz_p='required_val',  # required
    # perimout_o='t_proper_out',  # optional
    # mode_p=0,  # optional
    # area_p='0,1',  # optional
    # close_p=0,  # optional
    # clockwse_p='optional',  # optional
    # vplane_p=1,  # optional
    # dmax_p='optional',  # optional
    # tol_p=0,  # optional
    # reduce_p=0,  # optional
    # extend_p=0,  # optional
    # cross_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("proper execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_proper_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")